# Day 15 · Optimization
## Fast is not a property of code

**Databricks + Snowflake · 70-Hour Programme · DataTrends.tech**

---

Yesterday your pipeline started running by itself. So the question changes.

| | |
|---|---|
| Until yesterday | does it work? |
| From today | how long does it take, and what is it costing somebody? |

Every optimization you will ever do is one of three moves.

## Configuration

Change `MY_ID` to your own name or roll number, then run this cell.

In [ ]:
import re, time
from pyspark.sql import functions as F

MY_ID = "change_me"           # <-- put your name or roll number here

if MY_ID == "change_me":
    raise ValueError("Set MY_ID to your own name or roll number, then run this cell again.")

TAG     = re.sub(r"[^a-z0-9]+", "_", MY_ID.lower()).strip("_")
CATALOG = "workspace"
SCHEMA  = f"day15_{TAG}"

MANY  = f"{CATALOG}.{SCHEMA}.orders_many_files"
CLUS  = f"{CATALOG}.{SCHEMA}.orders_clustered"
PART  = f"{CATALOG}.{SCHEMA}.orders_partitioned"
SAVED = f"{CATALOG}.{SCHEMA}.daily_summary"

SOURCE = "samples.tpch.lineitem"
SMALL  = "samples.tpch.supplier"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

print(f"schema  : {CATALOG}.{SCHEMA}")
print(f"source  : {SOURCE}   (read-only, shared by everybody)")

### A stopwatch, and an honest warning about it

Timings on shared serverless compute move around. The first run of anything includes
start-up. So the helper below runs a query twice and keeps the **better** of the two.

Treat every number tonight as *roughly*, never as *exactly*. When two numbers are close,
the query plan is the evidence, not the stopwatch.

In [ ]:
def timed(label, fn, runs=2):
    """Run fn() a couple of times and report the best wall-clock time."""
    best = None
    out  = None
    for _ in range(runs):
        t0 = time.time()
        out = fn()
        dt = time.time() - t0
        best = dt if best is None else min(best, dt)
    print(f"{label:<38}{best:7.2f} s")
    return out


def detail(table):
    """The three numbers that describe a table's physical layout."""
    return (spark.sql(f"DESCRIBE DETAIL {table}")
                 .select("numFiles",
                         F.round(F.col("sizeInBytes") / 1024 / 1024, 1).alias("size_MB"),
                         "clusteringColumns",
                         "partitionColumns"))

---
# 1 · The three moves

There are only three ways to make a query faster. Everything else is a detail of one of them.

| Move | Means | Tonight |
|---|---|---|
| **Read less** | touch fewer bytes on disk | file layout, clustering |
| **Shuffle less** | move fewer rows between machines | join strategy |
| **Do it once** | stop recomputing the same thing | materialise the result |

Notice what is *not* on that list: writing cleverer code. A tidy query over a badly laid
out table loses to an ugly query over a well laid out one, every time.

---
# 2 · Read less

### First, build a table that is laid out badly

One million rows, deliberately spread across two hundred files. This is not a silly
example — it is exactly what a pipeline produces when it appends a small batch every few
minutes for a month.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {MANY}")

(spark.read.table(SOURCE)
      .select("l_orderkey", "l_suppkey", "l_quantity",
              "l_extendedprice", "l_discount", "l_shipdate", "l_shipmode")
      .limit(1_000_000)
      .repartition(200)
      .write.format("delta").mode("overwrite").saveAsTable(MANY))

display(detail(MANY))

Two hundred files for one million rows. Each file is a few hundred kilobytes.

Why that hurts: opening a file has a fixed cost, and it is paid two hundred times before
a single useful row is read. Small files are the most common performance problem in real
lakehouses, and nobody creates them on purpose.

In [ ]:
def scan_query():
    return (spark.table(MANY)
                 .where("l_shipmode = 'AIR'")
                 .agg(F.sum("l_extendedprice"))
                 .collect())

timed("200 small files", scan_query)

### OPTIMIZE — the same rows, fewer files

In [ ]:
display(spark.sql(f"OPTIMIZE {MANY}").select("metrics.numFilesAdded", "metrics.numFilesRemoved"))

In [ ]:
display(detail(MANY))

In [ ]:
timed("after OPTIMIZE", scan_query)

Same rows. Same query. Same answer. Only the physical layout changed.

If the two timings came out close, say so and look at the file count instead — that number
never lies, and on a small dataset the file count moves long before the clock does.

---
# 3 · Help it skip

Fewer files is good. **Reading none of them is better.**

Delta records the minimum and maximum of each column, for every file, in the transaction
log. So before opening a file, Spark can ask: could a row matching my filter possibly be
in here? If the answer is no, the file is never opened.

That is called **data skipping**, and it only works if related rows sit in the same file.
Which is what clustering is for.

### Liquid clustering

`CLUSTER BY` tells Delta which column to keep together. Databricks now recommends it for
**all new tables** — it replaces both Hive-style partitioning and `ZORDER`, and unlike
partitioning you can change your mind later without rewriting the table.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {CLUS}")

spark.sql(f"""
CREATE TABLE {CLUS}
CLUSTER BY (l_shipmode)
AS SELECT * FROM {MANY}
""")

spark.sql(f"OPTIMIZE {CLUS}")

display(detail(CLUS))

In [ ]:
def scan_clustered():
    return (spark.table(CLUS)
                 .where("l_shipmode = 'AIR'")
                 .agg(F.sum("l_extendedprice"))
                 .collect())

timed("clustered by l_shipmode", scan_clustered)

### Changing your mind is one line

This is the part that partitioning cannot do. No rewrite, no reload, no migration.

In [ ]:
spark.sql(f"ALTER TABLE {CLUS} CLUSTER BY (l_shipdate)")

display(detail(CLUS))

---
## YOUR TURN · 1

1. Write a query that filters `{CLUS}` on a single `l_shipdate`.
2. Time it before running `OPTIMIZE`, and again afterwards.
3. In one line: why did clustering on `l_shipdate` help *this* query and not the earlier one?

In [ ]:
# YOUR CODE HERE

---
# 4 · Partitioning — and why it is usually the wrong tool now

Partitioning splits a table into physical folders, one per value. It is in your syllabus,
it is in every older tutorial, and it is still the right answer occasionally.

Watch what it does to the layout.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {PART}")

(spark.table(MANY)
      .write.format("delta")
      .partitionBy("l_shipmode")
      .mode("overwrite")
      .saveAsTable(PART))

display(detail(PART))

In [ ]:
display(
    spark.table(PART)
         .groupBy("l_shipmode").count()
         .orderBy("l_shipmode")
)

`l_shipmode` has only a handful of values, so this behaves reasonably. Now imagine
partitioning by `l_shipdate` instead — thousands of dates, thousands of folders, each
holding a file too small to be worth opening. You would have rebuilt the small-file
problem on purpose.

| Partitioning | Liquid clustering |
|---|---|
| folders on disk, fixed at write time | file layout, adjustable any time |
| changing it means rewriting the table | one `ALTER TABLE` |
| high-cardinality column destroys it | high cardinality is fine |
| still fine for a very large table on a low-cardinality column | recommended for all new tables |

The honest summary: partition only when a table is genuinely huge and the column has few
values. Otherwise, cluster.

---
# 5 · Shuffle less

Reading is one cost. **Moving rows between machines is the other**, and it is usually the
bigger one. A join is where that happens.

Two ways to join a big table to a small one:

| | What happens |
|---|---|
| **Broadcast** | send the whole small table to every machine · no shuffle of the big one |
| **Sort-merge** | shuffle both tables so matching keys land together · expensive |

In [ ]:
small = spark.read.table(SMALL).select("s_suppkey", "s_name", "s_nationkey")

print("rows in the small table:", small.count())

In [ ]:
joined = (spark.table(MANY).alias("b")
               .join(small.alias("s"), F.col("b.l_suppkey") == F.col("s.s_suppkey"))
               .groupBy("s.s_nationkey")
               .agg(F.sum("b.l_extendedprice").alias("total")))

joined.explain(mode="formatted")

Look for the word **Broadcast** in the plan above. Spark saw that one side was small,
and chose to copy it everywhere rather than shuffle a million rows.

Nobody asked it to. That decision is Catalyst, plus Adaptive Query Execution looking at
the real sizes while the query runs.

In [ ]:
timed("join, Spark's own choice", lambda: joined.collect())

### Take the choice away from it

A hint forces the slower strategy, so you can see what Spark saved you from.

In [ ]:
forced = (spark.table(MANY).alias("b").hint("shuffle_merge")
               .join(small.alias("s"), F.col("b.l_suppkey") == F.col("s.s_suppkey"))
               .groupBy("s.s_nationkey")
               .agg(F.sum("b.l_extendedprice").alias("total")))

forced.explain(mode="formatted")

In [ ]:
timed("join, forced sort-merge", lambda: forced.collect())

Two lessons here, and the second one matters more.

1. Broadcasting a small table beats shuffling a big one.
2. **Spark already knew that.** The optimiser is usually right, and a hint is a way of
   telling it you know better. Most of the time, you do not.

Reach for a hint only when you have read the plan and can say exactly what is wrong with it.

---
# 6 · Do it once

The third move. If the same result is computed twice, one of those times was free money
given away.

In older Spark courses the answer here is `cache()`. Run the next cell.

In [ ]:
try:
    spark.table(MANY).cache().count()
    print("cache() worked on this compute.")
except Exception as e:
    print("cache() is not available here.")
    print("")
    print(str(e).split("\n")[0][:300])

### So what replaces it?

On serverless there is no cluster of your own to hold memory between queries, so caching
has nothing to live in. The modern answer is better anyway: **write the result down**.

| `cache()` | A Delta table |
|---|---|
| lives in memory | lives on disk |
| dies when the session ends | is still there tomorrow morning |
| only this notebook can use it | your job, your dashboard and your colleague can use it |
| invisible to anybody else | has a name, an owner and a history |

In [ ]:
daily = (spark.table(MANY)
              .groupBy("l_shipdate", "l_shipmode")
              .agg(F.sum("l_extendedprice").alias("revenue"),
                   F.count("*").alias("lines")))

daily.write.format("delta").mode("overwrite").saveAsTable(SAVED)

print("computed once, written down:", SAVED)

In [ ]:
timed("recompute from the big table",
      lambda: spark.table(MANY).groupBy("l_shipdate", "l_shipmode")
                   .agg(F.sum("l_extendedprice")).collect())

timed("read the saved summary",
      lambda: spark.table(SAVED).collect())

That is the whole medallion idea arriving from a different direction. Bronze, silver and
gold are not a naming convention — they are three levels of *work already done*, so that
nobody has to do it again.

---
## YOUR TURN · 2

1. Pick any aggregation you would put on a dashboard.
2. Time it directly against `{MANY}`.
3. Save it as a table, then time reading that table.
4. In one line: at what point does saving it stop being worth the storage?

In [ ]:
# YOUR CODE HERE

---
# 7 · What you cannot see from here

Being straight with you about the free tier.

| In a paid workspace | Here |
|---|---|
| Spark UI — stages, tasks, shuffle bytes, skew | not available |
| Query profile with per-operator timings | limited |
| Your own cluster, sized by you | serverless, sized for you |
| `cache()` and `persist()` | not available |

So your instrument tonight is `explain()` — the same tool from Day 8. That is not a
downgrade. The plan tells you *what Spark decided to do*, which is the question. The Spark
UI mostly tells you how long it then took to do it.

In your first job you will have both. Read the plan first anyway.

---
## Practice · on a table you did not build

`samples.tpch.orders` is available in every workspace and nobody has tuned it for you.

1. Write a query that filters it on `o_orderdate` and aggregates by `o_orderpriority`.
2. Read the plan. Find where the filter is applied.
3. Make your own clustered copy of the columns you need, and compare.
4. Write down the file count before and after — that number is your evidence.

In [ ]:
# YOUR CODE HERE

---
# The three moves, one more time

| Move | The question to ask | The tool |
|---|---|---|
| Read less | am I opening files I do not need? | `OPTIMIZE`, `CLUSTER BY` |
| Shuffle less | am I moving rows I do not need to move? | read the plan, check for Broadcast |
| Do it once | am I computing this again? | write it to a table |

And the sentence to leave with:

**Fast is not a property of your code. It is a property of how much work you avoided.**

---
## Clean up (optional)

Run this only when you are finished.

In [ ]:
# spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{SCHEMA} CASCADE")